In [0]:
%sql
--------------------------------------------------------------------------------------------------------------
--- Allscripts TouchWorks Dose Era
--- Adapted to Databricks SQL from Pure SQL dose_era written by Chris_Knoll:
--- https://gist.github.com/chrisknoll/c820cc12d833db2e3d1e
--- Source schema: _exponent.omop_tw
--- INTERVAL set to 30 days
--- NOTE: drug_concept_id must be non-zero for rows to be included.
---       Requires TW drug exposure mapping pipeline to complete first.
--------------------------------------------------------------------------------------------------------------

TRUNCATE TABLE _exponent.omop_tw.dose_era;

WITH cteDrugTarget AS (
    SELECT
        d.drug_exposure_id,
        d.person_id,
        c.concept_id AS drug_concept_id,
        d.dose_unit_concept_id AS unit_concept_id,
        d.effective_drug_dose AS dose_value,
        d.drug_exposure_start_date,
        d.days_supply,
        COALESCE(
            d.drug_exposure_end_date,
            CASE
                WHEN d.days_supply IS NOT NULL AND d.days_supply > 0
                    THEN date_add(d.drug_exposure_start_date, CAST(d.days_supply AS INT))
                ELSE NULL
            END,
            date_add(d.drug_exposure_start_date, 1)
        ) AS drug_exposure_end_date
    FROM _exponent.omop_tw.drug_exposure d
         JOIN _exponent.omop.concept_ancestor ca
           ON ca.descendant_concept_id = d.drug_concept_id
         JOIN _exponent.omop.concept c
           ON ca.ancestor_concept_id = c.concept_id
    WHERE c.vocabulary_id = 'RxNorm'
      AND c.concept_class_id = 'Ingredient'
      AND d.drug_concept_id != 0
      AND (d.days_supply IS NULL OR d.days_supply >= 0)
)

, cteEndDates AS (
    SELECT
        person_id,
        drug_concept_id,
        unit_concept_id,
        dose_value,
        date_add(event_date, -30) AS end_date
    FROM
    (
        SELECT
            person_id,
            drug_concept_id,
            unit_concept_id,
            dose_value,
            event_date,
            event_type,
            MAX(start_ordinal) OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS start_ordinal,
            ROW_NUMBER() OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
            ) AS overall_ord
        FROM
        (
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                drug_exposure_start_date AS event_date,
                -1 AS event_type,
                ROW_NUMBER() OVER (
                    PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                    ORDER BY drug_exposure_start_date
                ) AS start_ordinal
            FROM cteDrugTarget
            UNION ALL
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                date_add(drug_exposure_end_date, 30) AS event_date,
                1 AS event_type,
                NULL AS start_ordinal
            FROM cteDrugTarget
        ) RAWDATA
    ) e
    WHERE (2 * e.start_ordinal) - e.overall_ord = 0
)

, ctoDoseEraEnds AS (
    SELECT
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date,
        MIN(e.end_date) AS dose_era_end_date
    FROM cteDrugTarget dt
    JOIN cteEndDates e
      ON dt.person_id       = e.person_id
     AND dt.drug_concept_id = e.drug_concept_id
     AND dt.unit_concept_id = e.unit_concept_id
     AND dt.dose_value      <=> e.dose_value
     AND e.end_date >= dt.drug_exposure_start_date
    GROUP BY
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date
)

INSERT INTO _exponent.omop_tw.dose_era (
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_start_date,
    dose_era_end_date
)
SELECT
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    MIN(drug_exposure_start_date) AS dose_era_start_date,
    dose_era_end_date
FROM ctoDoseEraEnds
GROUP BY
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_end_date
ORDER BY
    person_id,
    drug_concept_id;

In [0]:
%sql
SELECT
  COUNT(*) AS dose_era_count,
  COUNT(DISTINCT person_id) AS persons_covered,
  COUNT(DISTINCT drug_concept_id) AS distinct_ingredients
FROM _exponent.omop_tw.dose_era